In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    print(BASE_DIR)


# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

Ambiente Locale rilevato. Procedo con l'esecuzione...
d:\GitHub repositories\crop-spatial-classification
✅ Collegamento ai dati riuscito! Cartella raw: d:\GitHub repositories\crop-spatial-classification\data\raw


In [ ]:
from pathlib import Path

# lista espandibile di anni target per l'estrazione dei dati
TARGET_YEARS = ["2023"]

main_directory = Path(f"{DATA_DIR}/raw/crops_types_yearly_capitanata_03035")
tifs_3035 = {}

for year in TARGET_YEARS:
    year_dir = main_directory / year
    if year_dir.is_dir():
        # trova tutti i .tif per questo anno
        tifs_3035[year] = [str(tif) for tif in year_dir.rglob("*.tif")]
        print(f"Anno {year}: trovati {len(tifs_3035[year])} file .tif")
    else:
        print(f"Cartella anno {year} non trovata in {main_directory}")

In [ ]:
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

tifs_4326 = {}

for year, file_paths in tifs_3035.items():
    year_file_list = []

    for row_path in file_paths:
        # costruisce il percorso del file riproiettato in EPSG:4326
        file_name = row_path.replace("03035", "4326").replace("raw", "processed")
        file_path = Path(file_name)
        
        # se il file riproiettato esiste già, salta la riproiezione
        if file_path.is_file():
            year_file_list.append(file_name)
            continue
        
        # crea la cartella di destinazione se non esiste
        file_path.parent.mkdir(parents=True, exist_ok=True)
        
        # apre il file GeoTIFF originale (EPSG:3035)
        raster_3035 = rioxarray.open_rasterio(row_path)

        # riproiezione in EPSG:4326, usando NEAREST per preservare i codici delle colture
        raster_4326 = raster_3035.rio.reproject("EPSG:4326", resampling=Resampling.nearest)

        # salva il file riproiettato
        raster_4326.rio.to_raster(file_name)
        raster_3035.close()

        year_file_list.append(file_name)
        
    tifs_4326[year] = year_file_list

print(f"\nRiproiezione completata per {sum(len(v) for v in tifs_4326.values())} file")

In [ ]:
from pathlib import Path
import rasterio
import numpy as np

# num. max di punti casuali da estrarre per ciascuna coltura da ogni file .tif
SAMPLES_PER_CLASS_PER_TIF = 10

# N.B. garantisce di ottenre gli stessi risultati casuali ad ogni esecuzione
np.random.seed(42)

points = []

# itera su tutti gli anni target e sui rispettivi .tif riproiettati
for year, file_list in tifs_4326.items():
    print(f"Campionamento in corso per l'anno {year} ({len(file_list)} file)")

    for tif_path in file_list:
        with rasterio.open(tif_path) as dataset:
            # legge la matrice 2D dei pixel (banda 1)
            crop_matrix = dataset.read(1)

            # crea una maschera per scartare i pixel nulli o nodata
            valid_mask = (crop_matrix > 0) & (crop_matrix < 65534)

            # trova l'elenco dei codici coltura presenti nella matrice
            crop_codes = np.unique(crop_matrix[valid_mask])

            for crop_code in crop_codes:
                # seleziona solo i pixel di questa coltura
                rows, cols = np.where(crop_matrix == crop_code)

                if len(rows) == 0:
                    continue

                # estrae a caso un campione di pixel senza ripetizioni
                n_samples = min(SAMPLES_PER_CLASS_PER_TIF, len(rows))
                indices = np.random.choice(len(rows), size=n_samples, replace=False)

                # converte i pixel selezionati in coordinate geografiche
                for index in indices:
                    row, c = rows[index], cols[index]
                    lon, lat = dataset.xy(row, c)

                    points.append({
                        "year": int(year),          # anno coltura
                        "lon": lon.item(),          # longitudine
                        "lat": lat.item(),          # latitudine
                        "code": crop_code.item()    # codice coltura
                    })

print(f"\nEstrazione completata! Totale punti estratti: {len(points)}")

In [ ]:
import json
from pathlib import Path

# percorso del file JSON dei punti estratti
points_path = Path(f'{DATA_DIR}/interim/points.json')
points_path.parent.mkdir(parents=True, exist_ok=True)

with open(points_path, "w", encoding="utf-8") as f:
    json.dump(points, f, indent=4, ensure_ascii=False)

print(f"✅ Salvati {len(points)} punti in: {points_path.resolve()}")

In [ ]:
import pandas as pd
import xml.etree.ElementTree as ET

# cerca i files .aux.xml che contengono i metadati
files = list(main_directory.rglob("*.aux.xml"))

if not files:
    raise FileNotFoundError(f"Nessun file .aux.xml trovato all'interno di: {main_directory}")

# prende il primo file trovato
file_path = files[0]
print(f"Trovato file .aux.xml: {file_path.name}")

tree = ET.parse(file_path)
root = tree.getroot()

legend = {}

# estrae il codice e il nome della coltura
for row in root.findall(".//Row"):
    fields = row.findall("F")
    
    crop_code = int(fields[0].text)     
    crop_name = fields[2].text
    
    legend[crop_code] = crop_name

In [ ]:
import json
from pathlib import Path

# percorso del file JSON della legenda
legend_path = Path(f'{DATA_DIR}/processed/legend.json')

with open(legend_path, "w", encoding="utf-8") as f:
    json.dump(legend, f, indent=4, ensure_ascii=False)

print(f"✅ Salvate {len(legend)} voci nella legenda: {legend_path.resolve()}")

In [ ]:
import pandas as pd

df_points = pd.read_json(f'{DATA_DIR}/interim/points.json')

df_points['crop'] = df_points['code'].map(legend).fillna("unknown")

summary = df_points.groupby(['code', 'crop']).size().reset_index(name='n_points')                                                                               
summary['percentage (%)'] = ((summary['n_points'] / len(df_points)) * 100).round(1)                                                                             
summary = summary.sort_values(by='n_points', ascending=False).reset_index(drop=True)

print(f"Riepilogo campionamento (Totale punti: {len(df_points)}):")
display(summary)